In [ ]:
"""
Sistema Sencillo de Gestión de Cuentas Bancarias
Autor: Abraham Zamudio
PIT349 Programación en PYTHON Intermedio (2025)
"""

import numpy as np
from datetime import datetime


from typing import Union, Dict, List
from dataclasses import dataclass
from enum import Enum
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

class TipoTransaccion(Enum):
    DEPOSITO = 1
    RETIRO = 2
    CONSULTA = 3

@dataclass
class Transaccion:
    id: str
    tipo: TipoTransaccion
    monto: float
    timestamp: datetime
    exitosa: bool

class CuentaBancaria:
    def __init__(self, titular: str, saldo_inicial: float = 0.0):
        self.titular = titular
        self._saldo = float(saldo_inicial)
        self._historial: List[Transaccion] = []
        self._id_counter = 0

        # Validación estadística de datos de entrada
        if saldo_inicial < 0:
            raise ValueError("El saldo inicial no puede ser negativo (p < 0.05 para pruebas de hipótesis)")

    def _generar_id_transaccion(self) -> str:
        """Genera ID único usando distribución uniforme discreta"""
        self._id_counter += 1
        return f"TXN{np.random.randint(10000, 99999)}_{self._id_counter}"

    def _validar_monto(self, monto: float) -> bool:
        """(límite operacional definido por política bancaria"""
        return monto > 0 and monto <= 1_000_000  # Límite práctico para transacciones

    def depositar(self, monto: float) -> bool:
        """Realiza depósito con validación probabilística"""
        if not self._validar_monto(monto):
            logging.warning(f"Intento de depósito inválido: {monto}")
            return False

        self._saldo += monto
        transaccion = Transaccion(
            id=self._generar_id_transaccion(),
            tipo=TipoTransaccion.DEPOSITO,
            monto=monto,
            timestamp=datetime.now(),
            exitosa=True
        )
        self._historial.append(transaccion)
        logging.info(f"Depósito exitoso: {monto}")
        return True

    def retirar(self, monto: float) -> bool:
        """Realiza retiro con control de saldo usando teoría de colas"""
        if not self._validar_monto(monto) or monto > self._saldo:
            logging.warning(f"Intento de retiro inválido: {monto}")
            return False

        self._saldo -= monto
        transaccion = Transaccion(
            id=self._generar_id_transaccion(),
            tipo=TipoTransaccion.RETIRO,
            monto=monto,
            timestamp=datetime.now(),
            exitosa=True
        )
        self._historial.append(transaccion)
        logging.info(f"Retiro exitoso: {monto}")
        return True

    def consultar_saldo(self) -> Dict[str, Union[float, List[Transaccion]]]:
        """Retorna saldo actual con análisis descriptivo"""
        transaccion = Transaccion(
            id=self._generar_id_transaccion(),
            tipo=TipoTransaccion.CONSULTA,
            monto=0,
            timestamp=datetime.now(),
            exitosa=True
        )
        self._historial.append(transaccion)

        return {
            'saldo_actual': self._saldo,
            'numero_transacciones': len(self._historial),
            'historial_reciente': self._historial[-5:] if self._historial else []
        }



    def analisis_estadistico(self) -> Dict[str, float]:
        """Análisis estadístico del historial transaccional"""
        montos = [t.monto for t in self._historial if t.tipo != TipoTransaccion.CONSULTA]

        if not montos:
            return {}

        return {
            'media': np.mean(montos),
            'desviacion_estandar': np.std(montos),
            'maximo': np.max(montos),
            'minimo': np.min(montos),
            'sumatoria_acumulada': np.sum(montos)
        }


# Implementación de patrón Factory para gestión de múltiples cuentas
class FabricaCuentas:
    _instancias = {}

    @classmethod
    def obtener_cuenta(cls, titular: str) -> CuentaBancaria:
        if titular not in cls._instancias:
            cls._instancias[titular] = CuentaBancaria(titular)
            logging.info(f"Nueva cuenta creada para: {titular}")
        return cls._instancias[titular]

In [ ]:
def main():
    cuenta = FabricaCuentas.obtener_cuenta("Cliente Ejemplo")

    while True:
        print("\n--- Sistema Bancario Científico ---")
        print("1. Depositar")
        print("2. Retirar")
        print("3. Consultar saldo")
        print("4. Análisis estadístico")
        print("5. Salir")

        opcion = input("Seleccione opción: ")

        try:
            if opcion == '1':
                monto = float(input("Monto a depositar: "))
                if cuenta.depositar(monto):
                    print("✓ Depósito exitoso")
                else:
                    print("✗ Error en depósito")

            elif opcion == '2':
                monto = float(input("Monto a retirar: "))
                if cuenta.retirar(monto):
                    print("✓ Retiro exitoso")
                else:
                    print("✗ Fondos insuficientes o monto inválido")

            elif opcion == '3':
                datos = cuenta.consultar_saldo()
                print(f"Saldo actual: ${datos['saldo_actual']:.2f}")
                print(f"Número de transacciones: {datos['numero_transacciones']}")

            elif opcion == '4':
                stats = cuenta.analisis_estadistico()
                if stats:
                    for k, v in stats.items():
                        print(f"{k.replace('_', ' ').title()}: {v:.2f}")
                else:
                    print("No hay datos para análisis")

            elif opcion == '5':
                print("¡Hasta luego!")
                break

            else:
                print("Opción inválida")

        except ValueError:
            print("Error: Ingrese valores numéricos válidos")
        except Exception as e:
            logging.error(f"Error inesperado: {str(e)}")
            print("Ocurrió un error interno")

if __name__ == "__main__":
    # Validación de entorno ejecución
    assert np.__version__ >= '1.20.0', "Requiere NumPy >= 1.20.0"
    main()